# 03: Probing Experiments — Main Results

**Goal**: Train probing classifiers on frozen embeddings and generate the core experimental results.

This notebook:
1. Loads cached embeddings from all models
2. Trains logistic regression probes for each relation type
3. Reports F1 scores per model × relation
4. Creates the main results comparison table

All experimental design principles:
- Stratified k-fold CV (k=5) to handle class imbalance
- Binary probing per relation (cleaner signal than multiclass)
- Report macro F1 (class-weighted) not just accuracy
- Keep regularization weak (C=1.0) to test the representation, not regularize it away

## 1. Setup and Load Cached Embeddings

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

# Notebooks run from notebooks/ — resolve paths relative to project root
PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
RESULTS_DIR = PROJECT_ROOT / "results"

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Available embeddings:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:30s} {size_mb:8.1f} MB")

Project root: /Users/norair/Documents/Code/Projects/spatial_probing
Cache directory: /Users/norair/Documents/Code/Projects/spatial_probing/results/embeddings
Available embeddings:
  clip_text_vsr_train.npy            15.0 MB
  sbert_vsr_train.npy                22.5 MB


## 2. Load Embeddings and Dataset Labels

In [ ]:
from datasets import load_dataset

print("Loading cached embeddings...")
sbert_embeddings = np.load(CACHE_DIR / "sbert_vsr_train.npy")
clip_text_embeddings = np.load(CACHE_DIR / "clip_text_vsr_train.npy")
clip_image_embeddings = np.load(CACHE_DIR / "clip_image_vsr_train.npy")
clip_concat_embeddings = np.load(CACHE_DIR / "clip_concat_vsr_train.npy")

print(f"Embedding shapes:")
print(f"  SBERT:            {sbert_embeddings.shape}")
print(f"  CLIP text:        {clip_text_embeddings.shape}")
print(f"  CLIP image:       {clip_image_embeddings.shape}")
print(f"  CLIP concat:      {clip_concat_embeddings.shape}")

print(f"\nLoading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]

relation_types = np.array([ex["relation"] for ex in train_data])
binary_labels = np.array([int(ex["label"]) for ex in train_data])

print(f"Dataset: {len(train_data)} examples")
print(f"Relation types: {len(np.unique(relation_types))} unique")
print(f"Label dist: {np.sum(binary_labels)} True, {len(binary_labels) - np.sum(binary_labels)} False")

## 3. Train Probes for Each Model

In [ ]:
from src.probing import probe_by_relation_type, model_comparison_table

models_to_probe = [
    ("sbert", sbert_embeddings),
    ("clip_text", clip_text_embeddings),
    ("clip_image", clip_image_embeddings),
    ("clip_concat", clip_concat_embeddings),
]

probe_results = {}
for model_name, embeddings in models_to_probe:
    print(f"Training {model_name} probe...")
    probe_results[model_name] = probe_by_relation_type(
        embeddings, relation_types, binary_labels, n_splits=5, C=1.0, min_samples=50
    )
    print(f"  Relations tested: {len(probe_results[model_name])}")
    print(f"  Mean F1: {probe_results[model_name]['f1_macro_mean'].mean():.3f}")

sbert_results = probe_results["sbert"]
clip_text_results = probe_results["clip_text"]
clip_image_results = probe_results["clip_image"]
clip_concat_results = probe_results["clip_concat"]

## 4. Main Results Table

In [ ]:
results_dict = {
    "sbert": sbert_results,
    "clip_text": clip_text_results,
    "clip_image": clip_image_results,
    "clip_concat": clip_concat_results,
}

comparison = model_comparison_table(results_dict)
print(comparison.head(15).to_string())

comparison.to_csv(RESULTS_DIR / "probing_results.csv", index=False)
print(f"\nSaved results table to results/probing_results.csv")

## 5. Summary Statistics

In [5]:
print("\n" + "="*70)
print("PROBING RESULTS SUMMARY")
print("="*70)

for model_name, results_df in results_dict.items():
    f1_scores = results_df["f1_macro_mean"].values
    print(f"\n{model_name.upper():15s}:")
    print(f"  Mean F1:       {f1_scores.mean():.3f} (±{f1_scores.std():.3f})")
    print(f"  Median F1:     {np.median(f1_scores):.3f}")
    print(f"  Min F1:        {f1_scores.min():.3f}")
    print(f"  Max F1:        {f1_scores.max():.3f}")
    print(f"  Relations:     {len(results_df)}")

print(f"\nCLIP text vs SBERT:")
common = set(sbert_results["relation"]) & set(clip_text_results["relation"])
sbert_f1   = {r: f for r, f in zip(sbert_results["relation"], sbert_results["f1_macro_mean"])}
clip_f1    = {r: f for r, f in zip(clip_text_results["relation"], clip_text_results["f1_macro_mean"])}
gains = [clip_f1[r] - sbert_f1[r] for r in common]
print(f"  Mean gain: {np.mean(gains):+.3f} (±{np.std(gains):.3f})")
print(f"  Relations where CLIP wins: {sum(g > 0 for g in gains)}/{len(gains)}")
print("\n" + "="*70)


PROBING RESULTS SUMMARY

SBERT          :
  Mean F1:       0.657 (±0.185)
  Median F1:     0.552
  Min F1:        0.497
  Max F1:        0.989
  Relations:     48

CLIP_TEXT      :
  Mean F1:       0.567 (±0.125)
  Median F1:     0.499
  Min F1:        0.493
  Max F1:        0.981
  Relations:     48

CLIP text vs SBERT:
  Mean gain: -0.090 (±0.130)
  Relations where CLIP wins: 1/48



## 7. Detailed Relation Analysis

In [ ]:
comparison_copy = comparison.copy()
comparison_copy["clip_text_gain"] = comparison_copy["clip_text_f1"] - comparison_copy["sbert_f1"]
comparison_copy["clip_image_gain"] = comparison_copy["clip_image_f1"] - comparison_copy["sbert_f1"]
comparison_copy["clip_concat_gain"] = comparison_copy["clip_concat_f1"] - comparison_copy["sbert_f1"]
comparison_copy["image_vs_text"] = comparison_copy["clip_image_f1"] - comparison_copy["clip_text_f1"]

print("Relations with LARGEST CLIP concat gains over SBERT:")
for _, row in comparison_copy.nlargest(5, "clip_concat_gain").iterrows():
    print(f"  {row['relation']:25s}: SBERT {row['sbert_f1']:.3f} → concat {row['clip_concat_f1']:.3f} ({row['clip_concat_gain']:+.3f})")

print(f"\nImage encoder vs text encoder (where does spatial signal live?):")
for _, row in comparison_copy.nlargest(5, "image_vs_text").iterrows():
    print(f"  {row['relation']:25s}: image {row['clip_image_f1']:.3f}, text {row['clip_text_f1']:.3f} ({row['image_vs_text']:+.3f})")

print(f"\nRelations where SBERT outperforms all CLIP conditions:")
sbert_best = comparison_copy[
    (comparison_copy["sbert_f1"] > comparison_copy["clip_text_f1"]) &
    (comparison_copy["sbert_f1"] > comparison_copy["clip_image_f1"]) &
    (comparison_copy["sbert_f1"] > comparison_copy["clip_concat_f1"])
]
for _, row in sbert_best.iterrows():
    print(f"  {row['relation']:25s}: SBERT {row['sbert_f1']:.3f}, text {row['clip_text_f1']:.3f}, image {row['clip_image_f1']:.3f}, concat {row['clip_concat_f1']:.3f}")

## 8. Save Detailed Results

In [11]:
for model_name, results_df in results_dict.items():
    csv_path = RESULTS_DIR / f"probing_results_{model_name}_detailed.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path.name}")

comparison_copy.to_csv(RESULTS_DIR / "probing_results_with_gains.csv", index=False)
print(f"Saved: probing_results_with_gains.csv")

Saved: probing_results_sbert_detailed.csv
Saved: probing_results_clip_text_detailed.csv
Saved: probing_results_with_gains.csv


## 9. Key Findings

In [12]:
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

print(f"\n1. OVERALL PERFORMANCE:")
for model_name in model_names:
    f1 = comparison[f"{model_name}_f1"].mean()
    print(f"   {model_name.upper():20s}: {f1:.1%} mean F1")

print(f"\n2. VISUAL GROUNDING BENEFIT (CLIP text vs SBERT):")
gain = comparison["clip_text_f1"].mean() - comparison["sbert_f1"].mean()
print(f"   Mean F1 gain: {gain:+.1%}")
print(f"   Relations where CLIP wins: {(comparison['clip_text_f1'] > comparison['sbert_f1']).sum()}/{len(comparison)}")

print(f"\n3. HARDEST RELATIONS (lowest CLIP text F1):")
for _, row in comparison.nsmallest(3, "clip_text_f1").iterrows():
    print(f"   {row['relation']:25s}: {row['clip_text_f1']:.3f}")

print(f"\n4. EASIEST RELATIONS (highest CLIP text F1):")
for _, row in comparison.nlargest(3, "clip_text_f1").iterrows():
    print(f"   {row['relation']:25s}: {row['clip_text_f1']:.3f}")

print("\n" + "="*70)


KEY FINDINGS

1. OVERALL PERFORMANCE:
   SBERT               : 65.7% mean F1
   CLIP_TEXT           : 56.7% mean F1

2. VISUAL GROUNDING BENEFIT (CLIP text vs SBERT):
   Mean F1 gain: -9.0%
   Relations where CLIP wins: 1/48

3. HARDEST RELATIONS (lowest CLIP text F1):
   next to                  : 0.493
   in                       : 0.494
   beside                   : 0.496

4. EASIEST RELATIONS (highest CLIP text F1):
   touching                 : 0.981
   on top of                : 0.951
   far away from            : 0.865

